# E8b + E9 + E8c: CNN mitigation, non-contrast shortcut check, mitigated fusion

**Revision plan, 00 log sections 21 (E8b, E8c; approved 17 Oct 2026) and 22 (E9; approved 18 Oct 2026).** Answers
Reviewer 2 point 6 (mitigation) and tests a possible shortcut found in E47 (the non-contrast phase of the cHCC-CCA batch
differs technically), which bears on R2-5, R2-3 and Reviewer 4's concern about pipeline artefacts.

- **Cells 1–4** are E5c's cells unchanged: they rebuild the corrected HCC-TACE-Seg crops (E6 rules) for evaluation.
- **E8b** — CNN trained with the phase-dropout protocol of E3 **plus scanner-variation augmentation** (blur, noise, gamma,
  in-plane resolution loss, slice-thickness blur, intensity scale), and **test-time intensity harmonisation** of external
  crops to the internal distribution (no labels). Variants: frozen D-CNN with harmonisation, new CNN, new CNN with harmonisation.
- **E9** — header facts of the internal phases by class, a technical-only check, and models **without the non-contrast
  phase** (CNN, radiomics, fusions). Interpretation fixed in the plan: a held-out macro-AUC drop > 0.03 or a clear fall in
  cHCC-CCA recall without the non-contrast phase means internal performance partly depends on the artefact.
- **E8c** — the pre-specified mitigated fusion (robust + ComBat radiomics + new CNN + age/sex), plus a clearly labelled
  post-hoc version without ComBat (ComBat was harmful in E8a).

Every model is trained on internal data only; each is evaluated once on MCT-LTDiag (203) and HCC-TACE-Seg (corrected).
**The pre-specified primary results (E6/E8.0) stay the primary results; the main CNN stays the phase-dropout CNN.**

**Runtime:** **A100 GPU**, about 2.5–3.5 hours (two CNN arms × 5 folds). **Resume:** if the runtime stops, set
`RESUME_RUN_ID` in Cell 1 to the stopped run's time (the part after `E8b_`) and run all cells again; trained folds are reused.
**When it finishes,** Cell 13 saves everything to Drive and shuts the runtime down, so no compute units are wasted (set `SHUTDOWN_WHEN_DONE = False` in Cell 13 to keep it). **Safety:** inputs are only read; outputs go to a new folder `liver_project/revision_2026-10/E8b_<time>/`.

In [ ]:
# Cell 1 — Setup (E5/E5c cell): installs, paths, input checks, output folder E8b_<time>
import os, sys, json, hashlib, platform, datetime, re, time, warnings, subprocess, shutil, tarfile
from pathlib import Path
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm==4.6.0', 'openpyxl', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E5_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
from concurrent.futures import ThreadPoolExecutor
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy.ndimage import binary_dilation
from scipy.stats import beta

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
EXT_DIR = PROJECT_DIR / 'external_validation' / 'HCC-TACE-Seg'
OLD_PREFIX = os.environ['LIVER_HISTORICAL_PREFIX']   # paths stored in the Week-10 manifest (folder moved 7 Sep 2026)
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']

INPUTS = {
    'manifest_v4': (EXT_DIR / 'nifti_v4_liver_adapter' / 'manifests' / 'external_manifest_for_week10_tumor_only_FILTERED_liver_adapter.csv',
                    'e428d3ad7b5928622e51b78c1c3346b9b168ef9e6b6205b00e33a61f9f84e5ad'),
    'dicom_catalog': (EXT_DIR / 'nifti' / 'logs' / 'dicom_catalog.csv', '515453cc76f824b0803a4d885e0065e395198e4ba3521c2df79b21762e395e62'),
    'historical_external': (PROJECT_DIR / 'results' / 'external_HCC_TACE_SEG_liver_adapter_fixed_103' / 'external_predictions_with_phase_audit.csv',
                            'a5e412a63778013130c90bb50ae37591e16e4d85964053d15abf83b1443424fb'),
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e5_v2_predictions': (REVISION_DIR / 'E5_20261017T062958Z' / 'E5_predictions.csv', '5100abe40d584c69872099121175d71d13a67ab407e684f8ae497391e12c1149'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, lightgbm, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'lightgbm': lightgbm.__version__,
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE)}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESUME_RUN_ID = None   # to continue a stopped run, put its time here, e.g. '20261018T020304Z'
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
OUT_DIR = REVISION_DIR / f'E8b_{RUN_ID}'
N_IO = 4 if IN_COLAB else 1
N_JOBS = os.cpu_count() if IN_COLAB else 1
OUT_DIR.mkdir(parents=False, exist_ok=bool(RESUME_RUN_ID))
(OUT_DIR / 'inputs').mkdir(exist_ok=True)
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Download the official TCIA clinical file and extract age and sex (stops if the format is unclear)
CLINICAL_URL = 'https://www.cancerimagingarchive.net/wp-content/uploads/HCC-TACE-Seg_clinical_data-V2.xlsx'
if IN_COLAB:
    import urllib.request
    clinical_path = OUT_DIR / 'inputs' / 'HCC-TACE-Seg_clinical_data-V2.xlsx'
    request = urllib.request.Request(CLINICAL_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=120) as response:
        clinical_path.write_bytes(response.read())
    sheets = pd.read_excel(clinical_path, sheet_name=None)
else:
    clinical_path = OUT_DIR / 'inputs' / 'clinical_test.csv'
    shutil.copy2(os.environ['E5_TEST_CLINICAL'], clinical_path)
    sheets = {'test': pd.read_csv(clinical_path)}
    if os.environ.get('E5_TEST_GLOSSARY'):
        sheets['glossary'] = pd.read_csv(os.environ['E5_TEST_GLOSSARY'])
input_hashes['tcia_clinical'] = sha256_of(clinical_path)

def normalized(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()

def find_clinical_table(sheets):
    for sheet_name, table in sheets.items():
        for column in table.columns:
            values = table[column].dropna().astype(str).str.strip()
            if len(values) and values.str.fullmatch(r'HCC_\d+').mean() > 0.9:
                return sheet_name, table, column
    return None, None, None

sheet_name, table, id_col = find_clinical_table(sheets)
age_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\bage\b', normalized(c))]
sex_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\b(sex|gender)\b', normalized(c))]
print('Sheets:', list(sheets))
if table is not None:
    print('Using sheet:', sheet_name, '| ID column:', id_col, '| age columns:', age_cols, '| sex columns:', sex_cols)
if table is None or len(age_cols) != 1 or len(sex_cols) != 1:
    for name, t in sheets.items():
        print(f'--- sheet {name}: columns', list(t.columns))
    raise RuntimeError('STOP — could not identify a unique patient-ID, age and sex column. '
                       'Send this output; the column choice will be set explicitly.')

SEX_MAP = {'m': 1.0, 'male': 1.0, 'f': 0.0, 'female': 0.0}     # internal coding: male = 1, female = 0

def sex_tokens(series):
    return series.astype(str).str.strip().str.lower().str.replace(r'^(\d+)\.0$', r'\1', regex=True)

def sex_codes_from_glossary(sheets, sex_col):
    """Numeric codes defined in the file's own glossary sheet (TCIA V2: 'Sex | 1=Male, 2=Female')."""
    for name, t in sheets.items():
        if name == sheet_name:
            continue
        for row in [list(t.columns)] + t.astype(str).values.tolist():
            cells = [str(v).strip() for v in row]
            if cells and normalized(cells[0]) == normalized(sex_col):
                text = ' '.join(cells[1:])
                pairs = re.findall(r'(\d+)\s*=\s*(male|female)\b', text, flags=re.I)
                if pairs:
                    return {code: SEX_MAP[word.lower()] for code, word in pairs}, f'sheet {name!r}: {cells[0]} | {text}'
    return {}, None

raw_sex = sex_tokens(table[sex_cols[0]].dropna())
print('Sex values in the file:', raw_sex.value_counts().to_dict())
SEX_CODING = 'text (male/female)'
if set(raw_sex) - set(SEX_MAP):
    glossary_codes, glossary_line = sex_codes_from_glossary(sheets, sex_cols[0])
    print('Sex coding stated in the file glossary:', glossary_line)
    if sorted(glossary_codes.values()) != [0.0, 1.0]:
        raise RuntimeError('STOP — numeric sex codes and no clear male/female definition in the glossary. Send this output.')
    SEX_MAP = {**SEX_MAP, **glossary_codes}
    SEX_CODING = f'numeric, from the glossary ({glossary_line}); mapped to male = 1, female = 0'
unmapped = sorted(set(raw_sex) - set(SEX_MAP))
if unmapped:
    raise RuntimeError(f'STOP — unrecognised sex codes {unmapped}. Send this output; the coding will be set explicitly.')
mapped = raw_sex.map(SEX_MAP)
print(f'Sex after mapping to the internal coding: male {int((mapped == 1).sum())}, female {int((mapped == 0).sum())}')
raw_age = pd.to_numeric(table[age_cols[0]], errors='coerce')
print(f'Age: {raw_age.notna().sum()} numeric values, range {raw_age.min()}–{raw_age.max()}')
if raw_age.notna().sum() and not (0 < raw_age.min() and raw_age.max() < 110):
    raise RuntimeError('STOP — age values outside 0–110. Send this output.')
clinical = pd.DataFrame({
    'patient_id': table[id_col].astype(str).str.strip().str.replace(r'HCC_(\d+)', lambda m: f'HCC_{int(m.group(1)):03d}', regex=True),
    'age': pd.to_numeric(table[age_cols[0]], errors='coerce'),
    'sex': sex_tokens(table[sex_cols[0]]).map(SEX_MAP),
}).dropna(subset=['patient_id'])
conflicts = clinical.groupby('patient_id').agg(age_n=('age', 'nunique'), sex_n=('sex', 'nunique'))
conflicting_ids = set(conflicts.index[(conflicts['age_n'] > 1) | (conflicts['sex_n'] > 1)])
clinical = clinical.groupby('patient_id', as_index=False).first()
clinical.loc[clinical['patient_id'].isin(conflicting_ids), ['age', 'sex']] = np.nan
print(f'Clinical rows: {len(clinical)} patients; conflicting duplicate rows set to missing: {len(conflicting_ids)}')

In [ ]:
# Cell 3 — External manifest (paths remapped to the new folder) and mixed-scan detection
manifest = pd.read_csv(INPUTS['manifest_v4'][0])
path_cols = [c for c in manifest.columns if re.match(r'(ct|mask|liver_mask)_(P|C1|C2|C3)$', c)]
def remap(value):
    if isinstance(value, str) and value.startswith(OLD_PREFIX):
        return str(PROJECT_DIR) + value[len(OLD_PREFIX):]
    return value
for column in path_cols:
    manifest[column] = manifest[column].map(remap)
referenced = {v for c in path_cols for v in manifest[c].dropna() if isinstance(v, str) and v.strip()}
missing_files = [v for v in referenced if not Path(v).exists()]
assert len(manifest) == 103, len(manifest)
if missing_files:
    raise RuntimeError(f'STOP — {len(missing_files)} manifest files not found, e.g. {missing_files[:3]}')

catalog = pd.read_csv(INPUTS['dicom_catalog'][0], usecols=['SeriesInstanceUID', 'Modality', 'ImagePositionPatient'])
catalog = catalog[catalog['Modality'] == 'CT'].copy()
catalog['z'] = catalog['ImagePositionPatient'].map(
    lambda s: round(float(re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)[2]), 1) if isinstance(s, str) else np.nan)
per_series = catalog.groupby('SeriesInstanceUID').agg(n=('z', 'size'), unique_z=('z', 'nunique'))
per_series['repeats'] = per_series['n'] / per_series['unique_z']

def series_uid(path):
    match = re.search(r'__([0-9.]+)__CT', str(path))
    return match.group(1) if match else None

mixed_rows = []
for _, row in manifest.iterrows():
    for phase in PHASE_ORDER:
        uid = series_uid(row.get(f'ct_{phase}'))
        if uid and uid in per_series.index and per_series.loc[uid, 'repeats'] >= 1.9:
            mixed_rows.append({'patient_id': row['patient_id'], 'channel': phase, 'series_uid': uid,
                               'files': int(per_series.loc[uid, 'n']), 'unique_positions': int(per_series.loc[uid, 'unique_z']),
                               'repeats': float(per_series.loc[uid, 'repeats'])})
mixed = pd.DataFrame(mixed_rows)
MIXED_IDS = set(mixed['patient_id']) if len(mixed) else set()
ALL_IDS = manifest['patient_id'].astype(str).tolist()
CLEAN_IDS = [p for p in ALL_IDS if p not in MIXED_IDS]
mixed.to_csv(OUT_DIR / 'E5_mixed_scan_inputs.csv', index=False)
print(f'Mixed-scan inputs: {len(mixed)} in {len(MIXED_IDS)} patients -> clean cohort {len(CLEAN_IDS)} patients')
if IN_COLAB and (len(mixed), len(MIXED_IDS)) != (16, 14):
    raise RuntimeError('STOP — mixed-scan count differs from the 17 Oct audit (16 inputs, 14 patients). Send this output.')

clinical_ext = pd.DataFrame({'patient_id': ALL_IDS}).merge(clinical, on='patient_id', how='left')
print(f'Observed age for {clinical_ext["age"].notna().sum()}/103, observed sex for {clinical_ext["sex"].notna().sum()}/103 '
      f'(male {int((clinical_ext["sex"] == 1).sum())}, female {int((clinical_ext["sex"] == 0).sum())})')
clinical_ext.to_csv(OUT_DIR / 'E5_external_clinical_joined.csv', index=False)

In [ ]:
# Cell 4 — Preprocessing: E5/Week-10g code for the gates, plus the axis-fixed variants (00 log 18)
TARGET_SPACING_MM = (1.0, 1.0, 1.0)
CROP_SIZE = (96, 96, 96)
TUMOR_DILATION_MM = 15
INPLANE_MM = 400.0 / 512                      # internal physical pixel size (E6 primary rule)

def resample_to_isotropic(arr, original_spacing, target_spacing=TARGET_SPACING_MM, is_mask=False):   # Week 10g
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(tuple(original_spacing))
    new_size = [int(round(sz * os_ / ts_)) for sz, os_, ts_ in zip(img.GetSize(), img.GetSpacing(), target_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0 if is_mask else -1024)
    out = sitk.GetArrayFromImage(resampler.Execute(img))
    return out.astype(np.uint8) if is_mask else out.astype(np.float32)

def window_historical(hu):          # Week 10g: W150/C50 -> -25..125 HU -> 0..255
    lo, hi = 50 - 150 / 2, 50 + 150 / 2
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

def window_corrected(hu):           # PLC-CECT release convention: -200..200 HU -> 0..255
    lo, hi = -200.0, 200.0
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

WINDOWS = {'historical_W150C50': window_historical, 'corrected_-200_200': window_corrected}

def crop_96_around_tumor(vol, tumor_union, size=CROP_SIZE):   # Week 10g
    if tumor_union.sum() == 0:
        cz, cy, cx = vol.shape[0] // 2, vol.shape[1] // 2, vol.shape[2] // 2
    else:
        cz, cy, cx = np.argwhere(tumor_union > 0).mean(axis=0).astype(int)
    dz, dy, dx = [s // 2 for s in size]
    out = np.zeros(size, dtype=vol.dtype)
    z0, z1 = max(0, cz - dz), min(vol.shape[0], cz + dz)
    y0, y1 = max(0, cy - dy), min(vol.shape[1], cy + dy)
    x0, x1 = max(0, cx - dx), min(vol.shape[2], cx + dx)
    oz0 = dz - (cz - z0); oy0 = dy - (cy - y0); ox0 = dx - (cx - x0)
    out[oz0:oz0 + (z1 - z0), oy0:oy0 + (y1 - y0), ox0:ox0 + (x1 - x0)] = vol[z0:z1, y0:y1, x0:x1]
    return out, (cz, cy, cx)

def zscore_nonzero(x):                                          # Week 10g / Week 4
    mask = x > 0
    if mask.sum() < 100:
        return x.astype(np.float32)
    vals = x[mask]
    out = np.zeros_like(x, dtype=np.float32)
    out[mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def read_nifti(path):
    if not isinstance(path, str) or not path.strip() or path.strip().lower() in {'nan', 'none', 'null'}:
        return None, None
    img = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32), img.GetSpacing()

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

# ---- orientation (E6 rule, 00 log 17.9): internal reference from the internal non-contrast scans ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254

def orientation_descriptor(ct0255_ijk, liver_ijk):
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

def decide(d):
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': bool(np.isfinite(off_i) and np.isfinite(off_j)
                                                              and max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03)}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')
def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}
with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
ok_int = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok_int['bone_j'] - .5) > abs(ok_int['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok_int['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok_int['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok_int['bulk_k'] < .5).mean()), 'n': int(len(ok_int))}
print('Internal orientation reference:', REF)
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')
# Expected orientation of these files (Week 10b wrote them with SimpleITK from DICOM: LPS, slices feet -> head); listed for the summary
EXPECTED = {'transpose': False, 'flip_i': False, 'flip_j': False, 'flip_k': True}

# ---- per patient: E5 outputs (gates) and the axis-fixed variants from the same Week-10g arrays ----
def load_resampled(row, target_for):
    raw, spacing = {}, None
    for phase in PHASE_ORDER:
        ct, sp = read_nifti(row.get(f'ct_{phase}'))
        if ct is not None and spacing is None:
            spacing = sp
        raw[phase] = (ct, read_nifti(row.get(f'mask_{phase}'))[0], read_nifti(row.get(f'liver_mask_{phase}'))[0])
    spacing = spacing or (1.0, 1.0, 1.0)
    target = target_for(spacing)
    ct_r, tm_r, lm_r = {}, {}, {}
    for phase, (ct, tm, lm) in raw.items():
        ct_r[phase] = None if ct is None else resample_to_isotropic(ct, spacing, target)
        tm_r[phase] = None if (ct is None or tm is None) else resample_to_isotropic((tm > 0.5).astype(np.uint8), spacing, target, is_mask=True)
        lm_r[phase] = None if (ct is None or lm is None) else resample_to_isotropic((lm > 0.5).astype(np.uint8), spacing, target, is_mask=True)
    masks_present = [m for m in tm_r.values() if m is not None]
    if not masks_present:
        return None
    ref_shape = masks_present[0].shape
    tumor_union = np.zeros(ref_shape, dtype=np.uint8)
    for m in masks_present:
        if m.shape == ref_shape:
            tumor_union = np.maximum(tumor_union, m)
    liver, liver_fallback = None, False
    for phase in ['C2', 'C1', 'P', 'C3']:
        lm = lm_r.get(phase)
        if lm is not None and lm.shape == ref_shape and lm.sum() > 0:
            liver = lm
            break
    if liver is None:
        liver, liver_fallback = binary_dilation(tumor_union > 0, iterations=TUMOR_DILATION_MM).astype(np.uint8), True
    return {'ct': ct_r, 'tm': tm_r, 'ref_shape': ref_shape, 'tumor_union': tumor_union, 'liver': liver, 'liver_fallback': liver_fallback}

def e5_outputs(r):                                              # E5 cell 4 loop, unchanged
    ct_r, tm_r, ref_shape, tumor_union, liver = r['ct'], r['tm'], r['ref_shape'], r['tumor_union'], r['liver']
    out = {'tumor_voxels': int(tumor_union.sum()), 'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8)}
    for window_name, window_fn in WINDOWS.items():
        cnn = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        rad = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        available = []
        for i, phase in enumerate(PHASE_ORDER):
            if ct_r[phase] is None or ct_r[phase].shape != ref_shape:
                continue
            crop, _ = crop_96_around_tumor(window_fn(ct_r[phase]) * (liver > 0), tumor_union)
            rad[i] = crop
            cnn[i] = zscore_nonzero(crop)
            available.append(phase)
            if tm_r[phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(tm_r[phase], tumor_union)[0]
        out[window_name] = {'cnn': cnn} if window_name == 'historical_W150C50' else {'cnn': cnn, 'rad': rad}
        out['available'] = available
    return out

def descriptor_of(r):
    for phase in ['P', 'C2', 'C1', 'C3']:                         # non-contrast first (bone not confused with contrast)
        if r['ct'][phase] is not None and r['ct'][phase].shape == r['ref_shape']:
            to_ijk = lambda a: np.transpose(a, (2, 1, 0))
            d = orientation_descriptor(window_corrected(to_ijk(r['ct'][phase])), to_ijk(r['liver']) > 0)
            if d is not None:
                d['phase_used'] = phase
            return d
    return None

def axis_fixed(r, o, rules):
    """(k, j, i) Week-10g arrays -> (i, j, k), matched to the internal orientation; Week-10g or Week-2 crop rules."""
    to = lambda a: apply_orientation(np.transpose(a, (2, 1, 0)), o)
    tu, lv = to(r['tumor_union']), to(r['liver'])
    out = {'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8), 'cnn': np.zeros((4, *CROP_SIZE), np.float32),
           'rad': np.zeros((4, *CROP_SIZE), np.float32 if rules == 'iso1mm' else np.uint8)}
    center = compute_center_of_mass(tu) or compute_center_of_mass(lv) or tuple(s // 2 for s in tu.shape)
    for i, phase in enumerate(PHASE_ORDER):
        ct = r['ct'][phase]
        if ct is None or ct.shape != r['ref_shape']:
            continue
        if rules == 'iso1mm':                                   # E5 rules, only the axes changed
            crop, _ = crop_96_around_tumor(window_corrected(to(ct)) * (lv > 0), tu)
            if r['tm'][phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(to(r['tm'][phase]), tu)[0]
        else:                                                   # E6 primary rules: Week-2 crop, 0-255 uint8
            crop = np.clip(crop_cube(window_corrected(to(ct)) * (lv > 0), center, CROP_SIZE[0]), 0, 255).astype(np.uint8)
            if r['tm'][phase] is not None:
                out['tumor'][i] = (crop_cube(to(r['tm'][phase]), center, CROP_SIZE[0]) > 0).astype(np.uint8)
        out['rad'][i] = crop
        out['cnn'][i] = zscore_nonzero(crop.astype(np.float32))
        out['available'].append(phase)
    out['tumor_in_crop'] = float(out['tumor'].max(axis=0).sum() / max(1, tu.sum()))
    return out

def header_orientation(direction):
    """Amendment 18.1: map index axes to the internal convention (axis 0 -> patient's left, axis 1 -> posterior,
    axis 2 -> feet) from the LPS direction cosines in the image header. None if not an axial acquisition."""
    D = np.array(direction, dtype=float).reshape(3, 3)          # column c = physical direction of index axis c
    p = [int(np.argmax(np.abs(D[:, c]))) for c in range(3)]
    s = [float(np.sign(D[p[c], c])) for c in range(3)]
    if p[2] != 2 or sorted(p[:2]) != [0, 1]:
        return None
    transpose = p[0] == 1
    a0, a1 = (1, 0) if transpose else (0, 1)
    return {'transpose': bool(transpose), 'flip_i': bool(s[a0] < 0), 'flip_j': bool(s[a1] < 0), 'flip_k': bool(s[2] > 0)}

def reference_ct_path(row):                                     # the CT whose spacing Week 10g uses (first available)
    for phase in PHASE_ORDER:
        path = row.get(f'ct_{phase}')
        if isinstance(path, str) and path.strip() and path.strip().lower() not in {'nan', 'none', 'null'}:
            return path
    return None

def header_direction(path):
    reader = sitk.ImageFileReader()
    reader.SetFileName(path)
    reader.ReadImageInformation()
    return reader.GetDirection()

def preprocess(row):
    pid = str(row['patient_id'])
    iso = load_resampled(row, lambda spacing: TARGET_SPACING_MM)
    if iso is None:
        return pid, None
    result = e5_outputs(iso)
    direction = header_direction(reference_ct_path(row))
    o = header_orientation(direction)
    d = descriptor_of(iso)
    anatomy = decide(d) if d is not None and np.isfinite(d['bone_i']) else {}
    result['orientation'] = {'header_axial': o is not None, 'direction': [round(float(v), 3) for v in direction],
                             **(o or {}), **{f'anatomy_{k}': v for k, v in anatomy.items()},
                             **(d or {}), 'liver_fallback': iso['liver_fallback']}
    if o is None:
        return pid, result                                       # stopped below, after all patients are listed
    result['axis_fixed_iso1mm'] = axis_fixed(iso, o, 'iso1mm')
    del iso
    e6 = load_resampled(row, lambda spacing: (INPLANE_MM, INPLANE_MM, float(spacing[2])))
    result['axis_fixed_E6rules'] = axis_fixed(e6, o, 'E6rules')
    return pid, result

PRE = {}
t0 = time.time()
rows = [row for _, row in manifest.iterrows()]
with ThreadPoolExecutor(N_IO) as pool:
    for n, (pid, result) in enumerate(pool.map(preprocess, rows), start=1):
        if result is None:
            raise RuntimeError(f'STOP — {pid} could not be preprocessed.')
        PRE[pid] = result
        if n % 10 == 0:
            print(f'  preprocessed {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

orient = pd.DataFrame({pid: PRE[pid]['orientation'] for pid in ALL_IDS}).T
orient.to_csv(OUT_DIR / 'E5c_orientation_decisions.csv')
NOT_AXIAL = orient.index[~orient['header_axial'].astype(bool)].tolist()
if NOT_AXIAL:
    raise RuntimeError(f'STOP — image header is not axial for {NOT_AXIAL[:10]}. Send this output.')
ORIENTATION_SUMMARY = {'source': 'image header (amendment 18.1)', 'liver_mask_fallback': int(orient['liver_fallback'].astype(bool).sum())}
for part in EXPECTED:
    applied = orient[part].astype(bool)
    clear = orient[f'anatomy_conf_{part}'].fillna(False).astype(bool) if f'anatomy_conf_{part}' in orient else pd.Series(False, index=orient.index)
    agree = (orient.loc[clear, f'anatomy_{part}'].astype(bool) == applied[clear]) if clear.any() else pd.Series(dtype=bool)
    ORIENTATION_SUMMARY[part] = {'applied_to': int(applied.sum()), 'anatomy_clear': int(clear.sum()),
                                 'anatomy_agrees_with_header': int(agree.sum())}
ORIENTATION_SUMMARY['directions'] = orient['direction'].astype(str).value_counts().head(5).to_dict()
print('Orientation decisions:', json.dumps(ORIENTATION_SUMMARY, indent=1))
VARIANTS = ['historical_W150C50', 'corrected_-200_200', 'axis_fixed_iso1mm', 'axis_fixed_E6rules']
print('Tumour fraction inside the crop (median):', {v: round(float(np.median([PRE[p][v]['tumor_in_crop'] for p in ALL_IDS])), 3)
                                                    for v in VARIANTS[2:]})

In [ ]:
# Cell 5 — E8b/E9 setup: extra inputs, internal crops, MCT-LTDiag crops (203), HCC-TACE-Seg corrected crops (Cell 4)
import random
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_classif
from sklearn.metrics import roc_auc_score
from scipy import ndimage
from scipy.stats import norm
import lightgbm as lgb

E6_RUN = REVISION_DIR / 'E6_20261017T114646Z'
E5C_RUN = REVISION_DIR / 'E5c_20261017T143217Z'
E8A_RUN = REVISION_DIR / 'E8a_20261017T162800Z'
E47_RUN = REVISION_DIR / 'E47_20261017T181102Z'
MCT_DIR = PROJECT_DIR / 'external_validation' / 'MCT-LTDiag'
NUM_EPOCHS = int(os.environ.get('E8B_TEST_EPOCHS', 40))       # 40 as in E3
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5                            # Week 4 / E3 settings
USE_AMP = DEVICE.type == 'cuda'
WORKERS = 6 if IN_COLAB else 0
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
INPUTS2 = {
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e3_original_predictions': (E3_DIR / 'E3A_original_cnn_predictions.csv', '5c02c3ebda984b6283c1137d85c17c171347b24ae1fac450dfc89c5513d0229b'),
    'mct_selected': (MCT_DIR / 'metadata' / 'selected_HCC_ICC_patients.csv', 'fdb8d8fa74d8211bbfe45ecb4820415e52a5183ce5d51bcfa4376cfa9ed11e20'),
    'e6_rad_primary': (E6_RUN / 'E6_radiomics_features_primary.csv', 'ac7077085e8f9e4bc458c14fe6a0cd8ba109d23898a81d8a3d70e289b8db7f56'),
    'e8a_rad_restored2': (E8A_RUN / 'E8_0_restored2_radiomics_features_primary.csv', '20b60144553843065046468e6d6815fb79daa6c2ed87f7aff175f521e65cedf3'),
    'e8a_e80_predictions': (E8A_RUN / 'E8_0_E6_203_predictions.csv', None),
    'e8a_mct_predictions': (E8A_RUN / 'E8a_mct203_predictions.csv', 'f5cb0f9062867f93ef252f06959ae97ad8e1bf7de03b05c27f4b53306432d5df'),
    'e8a_robust_features': (E8A_RUN / 'E8a_robust_features.json', 'd72e039e8c935a4d668284a71939450bc038bdf7b7877bf6e1d890ce96643aa1'),
    'e8a_internal_test': (E8A_RUN / 'E8a_internal_test.csv', '0d0cfa885197ab06573818aa9293d2f14a89246a4d2acb68224ae419861a5808'),
    'e5c_predictions': (E5C_RUN / 'E5c_predictions.csv', '45dd7f05598ba5cff5fb57091622d36afd332f55bf1b29a5e216eec0ed6999de'),
    'e5c_rad_E6rules': (E5C_RUN / 'E5c_radiomics_features_axis_fixed_E6rules.csv', 'b191f36146407e3d980e25ef3b3c72c4478fb90aee6f241e2c424809add4f18b'),
    'e47_truncation': (E47_RUN / 'E4a_truncation_per_patient_phase.csv', '74c616ac0fdd73fd844b8058e1f355a06fcb225636db2ff9b38b1ff8d6af2d4e'),
}
problems2 = []
for key, (path, expected) in INPUTS2.items():
    if not path.exists():
        problems2.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems2 if IN_COLAB else []).append(f'changed: {path.name}')
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems2.append(f'missing or changed size: {tar_path}')
if problems2:
    raise RuntimeError('STOP — E8b input check failed:\n' + '\n'.join(problems2))

# ---- internal crops (processed.tar, as E3) ----
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E8B_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
CT, LABEL, LIVER_CROP = {}, {}, {}
for pid in dev_ids + test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        CT[pid], LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])
        LIVER_CROP[pid] = z['liver_mask'].astype(bool)

# ---- MCT-LTDiag: primary crops of all 203 (E6 run + the 2 restored by E8.0) ----
SEL = pd.read_csv(INPUTS2['mct_selected'][0], dtype=str)
MCT_IDS = SEL['patient_id'].tolist()
MCT_CT = {}
for pid in MCT_IDS:
    path = E6_RUN / 'crops_primary' / f'{pid}.npz'
    if not path.exists():
        path = E8A_RUN / 'crops_primary_restored2' / f'{pid}.npz'
    with np.load(path, allow_pickle=False) as z:
        MCT_CT[pid] = z['ct'].astype(np.uint8)
if len(MCT_CT) != 203 and IN_COLAB:
    raise RuntimeError(f'STOP — {len(MCT_CT)} MCT-LTDiag crops found, expected 203.')
Y_MCT = (SEL.set_index('patient_id').loc[MCT_IDS, 'type'] == 'HCC').astype(int).to_numpy()
AGE_MCT = SEL.set_index('patient_id').loc[MCT_IDS, 'patient_age'].str.extract(r'0*(\d+)')[0].astype(float).to_numpy()
SEX_MCT = SEL.set_index('patient_id').loc[MCT_IDS, 'patient_sex'].str.strip().str.upper().map({'M': 1.0, 'F': 0.0}).to_numpy()

# ---- HCC-TACE-Seg corrected (E6 rules) crops from Cell 4 ----
TACE_IDS, TACE_CLEAN = list(ALL_IDS), list(CLEAN_IDS)
TACE_CT = {pid: PRE[pid]['axis_fixed_E6rules']['rad'].astype(np.uint8) for pid in TACE_IDS}
TACE_CLIN = clinical_ext.set_index('patient_id').loc[TACE_IDS]
del PRE
print(f'Internal crops {len(CT)}, MCT-LTDiag crops {len(MCT_CT)}, HCC-TACE-Seg crops {len(TACE_CT)} (clean {len(TACE_CLEAN)})')

In [ ]:
# Cell 6 — Training code: E3 phase-dropout protocol, plus scanner-variation augmentation and the no-non-contrast option
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def augment(vol4):                            # Week 4 / E3, unchanged
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept(allowed):
    """E3 phase dropout over the allowed channels: all allowed with p = 0.5, else drop each with p = 0.5, keep >= 1."""
    if random.random() < 0.5:
        return tuple(allowed)
    while True:
        kept = tuple(c for c in allowed if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

def domain_randomise(raw4):
    """Scanner-variation augmentation (plan 21, E8b-2) on the 0-255 crop before normalisation; one draw for all phases."""
    x = raw4.astype(np.float32)
    mask = x > 0
    if random.random() < 0.5:
        sigma = random.uniform(0.0, 1.5)
        x = np.stack([ndimage.gaussian_filter(x[c], sigma) for c in range(4)])
    if random.random() < 0.5:
        x = x + np.random.normal(0.0, random.uniform(0.0, 10.0), x.shape).astype(np.float32)
    if random.random() < 0.5:
        gamma = random.uniform(0.7, 1.5)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** gamma
    if random.random() < 0.5:
        f = random.uniform(0.5, 1.0)
        down = [ndimage.zoom(x[c], (f, f, 1.0), order=1) for c in range(4)]
        x = np.stack([ndimage.zoom(d, (96 / d.shape[0], 96 / d.shape[1], 1.0), order=1)[:96, :96, :96] for d in down])
    if random.random() < 0.5:
        sk = random.uniform(0.0, 2.0)
        x = np.stack([ndimage.gaussian_filter(x[c], (0, 0, sk)) for c in range(4)])
    if random.random() < 0.5:
        x = x * random.uniform(0.9, 1.1)
    return (np.clip(x, 0, 255) * mask).astype(np.float32)

class CropDataset(Dataset):
    def __init__(self, ids, arrays, train=False, phase_dropout=False, dr=False, allowed=(0, 1, 2, 3), kept=None):
        self.ids, self.arrays, self.train, self.phase_dropout, self.dr = list(ids), arrays, train, phase_dropout, dr
        self.allowed, self.kept = tuple(allowed), tuple(kept) if kept is not None else tuple(allowed)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        raw = self.arrays[pid].astype(np.float32)
        if self.train and self.dr:
            raw = domain_randomise(raw)
        vol4 = zscore4(raw)
        if self.train:
            vol4 = augment(vol4)
        kept = sample_kept(self.allowed) if (self.train and self.phase_dropout) else self.kept
        if kept != (0, 1, 2, 3):
            vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL.get(pid, 0), pid

def seed_worker(worker_id):                  # each loader worker gets its own numpy stream (torch seeds python/torch per worker)
    np.random.seed(torch.initial_seed() % 2 ** 32)

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

@torch.no_grad()
def predict(model, ids, arrays, kept=(0, 1, 2, 3), batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids, arrays, kept=kept), batch_size=batch_size, shuffle=False,
                        num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=P3)

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)

def train_fold(arm, cfg, fold):
    path = OUT_DIR / arm / f'fold_{fold}.pt'
    if path.exists():
        print(f'  {arm} fold {fold}: already trained, loading.')
        return
    path.parent.mkdir(exist_ok=True)
    outer_train = list(fold_splits[fold]['train_ids'])
    inner_train, inner_val = train_test_split(outer_train, test_size=0.2, random_state=SEED + fold,
                                              stratify=[LABEL[p] for p in outer_train])
    set_seeds(SEED + fold)
    loader = DataLoader(CropDataset(inner_train, CT, train=True, phase_dropout=True, dr=cfg['dr'], allowed=cfg['allowed']),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=USE_AMP, drop_last=True,
                        persistent_workers=WORKERS > 0, worker_init_fn=seed_worker)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(inner_train).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_inner = np.array([LABEL[p] for p in inner_val])
    best_auc, best_state, best_epoch, history = -1.0, None, -1, []
    for epoch in range(NUM_EPOCHS):
        model.train()
        started, loss_sum, n = time.time(), 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()
        inner_auc = macro_auc(y_inner, predict(model, inner_val, CT, kept=cfg['allowed']).loc[inner_val].to_numpy())
        history.append({'epoch': epoch, 'train_loss': loss_sum / max(n, 1), 'inner_val_auc': inner_auc, 'seconds': time.time() - started})
        if inner_auc > best_auc:
            best_auc, best_epoch = inner_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({'state_dict': best_state, 'arm': arm, 'fold': fold, 'best_epoch': best_epoch, 'best_inner_val_auc': best_auc,
                'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, 'epochs': NUM_EPOCHS}, path)
    pd.DataFrame(history).to_csv(OUT_DIR / arm / f'fold_{fold}_history.csv', index=False)
    print(f'  {arm} fold {fold}: best inner-val AUC {best_auc:.4f} at epoch {best_epoch} ({sum(h["seconds"] for h in history) / 60:.1f} min)', flush=True)
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

In [ ]:
# Cell 7 — Train the two new arms (5 folds each, same folds, seeds and inner split rule as E3)
ARMS = {'DR_phase_dropout': {'dr': True, 'allowed': (0, 1, 2, 3)},          # E8b-2: scanner-variation augmentation
        'noP_phase_dropout': {'dr': False, 'allowed': (1, 2, 3)}}            # E9b: non-contrast phase never seen
t0 = time.time()
for arm, cfg in ARMS.items():
    print(arm)
    for fold in range(5):
        train_fold(arm, cfg, fold)
print(f'Training done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 8 — Predictions: internal (out-of-fold, held-out, missing-phase patterns), MCT-LTDiag and HCC-TACE-Seg; harmonisation
PATTERNS = {'complete': (0, 1, 2, 3), 'drop_P': (1, 2, 3), 'drop_C1': (0, 2, 3), 'drop_C2': (0, 1, 3), 'drop_C3': (0, 1, 2),
            'P+C2': (0, 2), 'C2_only': (2,)}
NOP_PATTERNS = {'noP': (1, 2, 3), 'noP_drop_C3': (1, 2), 'C2_only': (2,)}
MODELS = {'phase_dropout_CNN': [load_state(MODEL_FILES[f'd_{f}']) for f in range(5)],
          'DR_CNN': [load_state(OUT_DIR / 'DR_phase_dropout' / f'fold_{f}.pt') for f in range(5)],
          'noP_CNN': [load_state(OUT_DIR / 'noP_phase_dropout' / f'fold_{f}.pt') for f in range(5)]}

def oof_and_test(models, kept):
    oof = pd.concat([predict(models[f], [p for p in fold_splits[f]['val_ids']], CT, kept=kept) for f in range(5)])
    test = sum(predict(m, test_ids, CT, kept=kept).loc[test_ids] for m in models) / len(models)
    return oof.loc[dev_ids], test

def external(models, arrays, ids, kept):
    return sum(predict(m, ids, arrays, kept=kept).loc[ids] for m in models) / len(models)

# ---- test-time intensity harmonisation (E8b-1): histogram matching to the pooled internal development distribution ----
LEVELS = np.linspace(0, 1, 1001)
REF_Q = []
for c in range(4):
    pooled = np.concatenate([CT[p][c][CT[p][c] > 0][::5] for p in dev_ids]).astype(np.float32)
    REF_Q.append(np.quantile(pooled, LEVELS))
def harmonise(x4):
    out = x4.astype(np.float32).copy()
    for c in range(4):
        m = out[c] > 0
        if m.sum() < 100:
            continue
        v = out[c][m]
        ranks = (np.argsort(np.argsort(v)) + 0.5) / len(v)
        out[c][m] = np.interp(ranks, LEVELS, REF_Q[c])
    return np.clip(out, 0, 255).astype(np.float32)
MCT_H = {p: harmonise(MCT_CT[p]) for p in MCT_IDS}
TACE_H = {p: harmonise(TACE_CT[p]) for p in TACE_IDS}

INT = {}                                       # (model, pattern) -> (oof 222, test 56)
for name, patterns in [('phase_dropout_CNN', {'complete': PATTERNS['complete']}), ('DR_CNN', PATTERNS), ('noP_CNN', NOP_PATTERNS)]:
    for pattern, kept in patterns.items():
        INT[(name, pattern)] = oof_and_test(MODELS[name], kept)
EXT_MCT, EXT_TACE = {}, {}
for name, kept in [('phase_dropout_CNN', (0, 1, 2, 3)), ('DR_CNN', (0, 1, 2, 3)), ('noP_CNN', (1, 2, 3))]:
    EXT_MCT[name] = external(MODELS[name], MCT_CT, MCT_IDS, kept)
    EXT_TACE[name] = external(MODELS[name], TACE_CT, TACE_IDS, kept)
for name in ['phase_dropout_CNN', 'DR_CNN']:
    EXT_MCT[f'{name} + harmonisation'] = external(MODELS[name], MCT_H, MCT_IDS, (0, 1, 2, 3))
    EXT_TACE[f'{name} + harmonisation'] = external(MODELS[name], TACE_H, TACE_IDS, (0, 1, 2, 3))

# ---- gates: the frozen D-CNN must reproduce E3 (internal), E8.0 (MCT-LTDiag) and E5c (HCC-TACE-Seg) ----
d_saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_saved_test = d_saved.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, P3]
e80 = pd.read_csv(INPUTS2['e8a_e80_predictions'][0], dtype={'patient_id': str})
e80 = e80[e80['variant'] == 'primary']
e5c_saved = pd.read_csv(INPUTS2['e5c_predictions'][0], dtype={'patient_id': str})
e5c_saved = e5c_saved[e5c_saved['variant'] == 'axis_fixed_E6rules']
GATES = {'internal_D_vs_E3': float(np.abs(INT[('phase_dropout_CNN', 'complete')][1].to_numpy() - d_saved_test.to_numpy()).max()),
         'MCT_D_vs_E8_0': float(np.abs(EXT_MCT['phase_dropout_CNN'].to_numpy()
                                       - e80[e80['model'] == 'phase_dropout_CNN'].set_index('patient_id').loc[MCT_IDS, P3].to_numpy()).max()),
         'TACE_D_vs_E5c': float(np.abs(EXT_TACE['phase_dropout_CNN'].to_numpy()
                                       - e5c_saved[e5c_saved['model'] == 'phase_dropout_CNN'].set_index('patient_id').loc[TACE_IDS, P3].to_numpy()).max())}
print('Gates (max probability differences, must be < 0.02):', GATES)
if IN_COLAB and max(GATES.values()) > 0.02:
    raise RuntimeError('STOP — the frozen phase-dropout CNN does not reproduce its saved predictions. Send this output.')

In [ ]:
# Cell 9 — Radiomics: without non-contrast features (E9b) and the robust model (for E8c), E2 recipe unchanged
K_TOP_FEATURES, CORR_THRESHOLD = 30, 0.95
def select_features_train(X_tr, y_tr, feature_names, k=K_TOP_FEATURES, corr_thresh=CORR_THRESHOLD):   # Week 5 / E2
    stds = np.nanstd(X_tr, axis=0)
    keep = stds > 1e-10
    X_kept = X_tr[:, keep]
    kept_idx_global = np.where(keep)[0]
    X_nan_to_num = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    corr = np.abs(np.corrcoef(X_nan_to_num.T))
    remove = np.zeros(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        if remove[i]:
            continue
        for j in range(i + 1, corr.shape[0]):
            if remove[j]:
                continue
            if corr[i, j] > corr_thresh:
                remove[j] = True
    kept_idx_after_corr = kept_idx_global[~remove]
    X_for_f = SimpleImputer(strategy='median').fit_transform(X_tr[:, kept_idx_after_corr])
    F, _ = f_classif(X_for_f, y_tr)
    order = np.argsort(F)[::-1]
    final_idx = kept_idx_after_corr[order[:min(k, len(order))]]
    return final_idx, [feature_names[i] for i in final_idx]

def train_one_fold_radiomics(fold_idx, train_ids, val_ids, df_features, feature_cols):            # Week 5 / E2
    df_tr = df_features[df_features['patient_id'].isin(train_ids)].reset_index(drop=True)
    df_va = df_features[df_features['patient_id'].isin(val_ids)].reset_index(drop=True)
    X_tr_all = df_tr[feature_cols].values.astype(np.float32)
    y_tr = df_tr['label'].values
    X_va_all = df_va[feature_cols].values.astype(np.float32)
    sel_idx, sel_names = select_features_train(X_tr_all, y_tr, feature_cols)
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_all[:, sel_idx])
    X_va = imp.transform(X_va_all[:, sel_idx])
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, num_leaves=15, min_child_samples=5,
                             reg_alpha=0.1, reg_lambda=0.1, class_weight='balanced', random_state=42, verbose=-1)
    clf.fit(X_tr, y_tr)
    return {'fold': fold_idx, 'probs': clf.predict_proba(X_va), 'patient_ids': df_va['patient_id'].tolist(),
            'sel_idx': sel_idx, 'sel_names': sel_names, 'imputer': imp, 'scaler': scaler, 'clf': clf}

def run_radiomics(df_features, feature_cols):
    folds = [train_one_fold_radiomics(f, s['train_ids'], s['val_ids'], df_features, feature_cols) for f, s in enumerate(fold_splits)]
    oof = pd.concat([pd.DataFrame(r['probs'], index=r['patient_ids'], columns=P3) for r in folds]).loc[dev_ids]
    return folds, oof, apply_fold_models(folds, df_features.set_index('patient_id').loc[test_ids], feature_cols)

def apply_fold_models(folds, features, feature_cols):
    X = features.reindex(columns=feature_cols).to_numpy(dtype=np.float32)
    probs = np.mean([r['clf'].predict_proba(r['scaler'].transform(r['imputer'].transform(X[:, r['sel_idx']]))) for r in folds], axis=0)
    return pd.DataFrame(probs, index=features.index, columns=P3)

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
POOL = joblib.load(MODEL_FILES['rad_0'])['feature_pool']
NOP_POOL = [c for c in POOL if not c.startswith('P_')]
ROBUST = json.loads(INPUTS2['e8a_robust_features'][0].read_text())
NOP_FOLDS, NOP_OOF, NOP_TEST = run_radiomics(internal, NOP_POOL)
ROB_FOLDS, ROB_OOF, ROB_TEST = run_radiomics(internal, ROBUST)
y_test = np.array([LABEL[p] for p in test_ids])
saved_robust = pd.read_csv(INPUTS2['e8a_internal_test'][0]).set_index('model').loc['radiomics_robust', 'internal_test_macro_auc']
GATES['robust_radiomics_vs_E8a'] = abs(macro_auc(y_test, ROB_TEST.to_numpy()) - float(saved_robust))
print(f'No-non-contrast radiomics: {len(NOP_POOL)} features | robust radiomics reproduces E8a: difference {GATES["robust_radiomics_vs_E8a"]:.1e}')
if IN_COLAB and GATES['robust_radiomics_vs_E8a'] > 1e-6:
    raise RuntimeError('STOP — the robust radiomics model does not reproduce E8a. Send this output.')

mct_feat = pd.concat([pd.read_csv(INPUTS2['e6_rad_primary'][0], index_col=0), pd.read_csv(INPUTS2['e8a_rad_restored2'][0], index_col=0)])
mct_feat.index = mct_feat.index.astype(str)
mct_feat = mct_feat.loc[MCT_IDS]
tace_feat = pd.read_csv(INPUTS2['e5c_rad_E6rules'][0], index_col=0)
tace_feat.index = tace_feat.index.astype(str)
tace_feat = tace_feat.loc[TACE_IDS]
e8a_mct = pd.read_csv(INPUTS2['e8a_mct_predictions'][0], dtype={'patient_id': str})
RAD = {   # name -> (oof 222, test 56, MCT 203, HCC-TACE-Seg 103)
    'radiomics_noshape': (pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids, ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']].set_axis(P3, axis=1),
                          pd.read_csv(INPUTS2['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']].set_axis(P3, axis=1),
                          e80[e80['model'] == 'radiomics_noshape'].set_index('patient_id').loc[MCT_IDS, P3],
                          e5c_saved[e5c_saved['model'] == 'radiomics_noshape'].set_index('patient_id').loc[TACE_IDS, P3]),
    'radiomics_noP': (NOP_OOF, NOP_TEST, apply_fold_models(NOP_FOLDS, mct_feat, NOP_POOL), apply_fold_models(NOP_FOLDS, tace_feat, NOP_POOL)),
    'radiomics_robust_ComBat': (ROB_OOF, ROB_TEST, e8a_mct[e8a_mct['model'] == 'radiomics_robust_ComBat'].set_index('patient_id').loc[MCT_IDS, P3],
                                apply_fold_models(ROB_FOLDS, tace_feat, ROBUST)),     # ComBat not applied to the HCC-only cohort (E8a rule)
    'radiomics_robust': (ROB_OOF, ROB_TEST, e8a_mct[e8a_mct['model'] == 'radiomics_robust'].set_index('patient_id').loc[MCT_IDS, P3],
                         apply_fold_models(ROB_FOLDS, tace_feat, ROBUST)),
}

In [ ]:
# Cell 10 — Fusions on internal out-of-fold predictions (meta-learners exactly as E2/E5/E6), applied once externally
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
oof_meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids].reset_index()
test_meta = pd.read_csv(INPUTS2['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids].reset_index()

def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X

def meta_folds(columns, oof):
    fitted, oof_p = [], np.full((len(oof), 3), np.nan)
    for f in range(5):
        tr, va = oof['fold'].astype(int).ne(f).to_numpy(), oof['fold'].astype(int).eq(f).to_numpy()
        X = prepare(oof.loc[tr, columns].to_numpy())
        imp = SimpleImputer(strategy='median').fit(X)
        sc = StandardScaler().fit(imp.transform(X))
        m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof.loc[tr, 'label'].astype(int))
        oof_p[va] = m.predict_proba(sc.transform(imp.transform(prepare(oof.loc[va, columns].to_numpy()))))
        fitted.append((imp, sc, m))
    return fitted, pd.DataFrame(oof_p, index=oof['patient_id'], columns=P3)

def apply_meta(fitted, frame, columns):
    return pd.DataFrame(np.mean([m.predict_proba(s.transform(i.transform(prepare(frame[columns].to_numpy())))) for i, s, m in fitted], axis=0),
                        index=frame.index, columns=P3)

CNN_SOURCES = {   # name -> (oof 222, test 56, MCT 203, TACE 103)
    'phase_dropout_CNN': (*INT[('phase_dropout_CNN', 'complete')], EXT_MCT['phase_dropout_CNN'], EXT_TACE['phase_dropout_CNN']),
    'phase_dropout_CNN + harmonisation': (*INT[('phase_dropout_CNN', 'complete')], EXT_MCT['phase_dropout_CNN + harmonisation'], EXT_TACE['phase_dropout_CNN + harmonisation']),
    'DR_CNN': (*INT[('DR_CNN', 'complete')], EXT_MCT['DR_CNN'], EXT_TACE['DR_CNN']),
    'DR_CNN + harmonisation': (*INT[('DR_CNN', 'complete')], EXT_MCT['DR_CNN + harmonisation'], EXT_TACE['DR_CNN + harmonisation']),
    'noP_CNN': (*INT[('noP_CNN', 'noP')], EXT_MCT['noP_CNN'], EXT_TACE['noP_CNN']),
}
FUSION_SPECS = [   # (label, cnn source, radiomics source, role)
    ('[ref] phase_dropout_CNN + radiomics_noshape', 'phase_dropout_CNN', 'radiomics_noshape', 'reference'),
    ('E8b: phase_dropout_CNN + harmonisation + radiomics_noshape', 'phase_dropout_CNN + harmonisation', 'radiomics_noshape', 'E8b'),
    ('E8b: DR_CNN + radiomics_noshape', 'DR_CNN', 'radiomics_noshape', 'E8b'),
    ('E8b: DR_CNN + harmonisation + radiomics_noshape', 'DR_CNN + harmonisation', 'radiomics_noshape', 'E8b'),
    ('E9: noP_CNN + radiomics_noP', 'noP_CNN', 'radiomics_noP', 'E9'),
    ('E8c (pre-specified): DR_CNN + radiomics_robust_ComBat', 'DR_CNN', 'radiomics_robust_ComBat', 'E8c'),
    ('E8c (post hoc): DR_CNN + radiomics_robust', 'DR_CNN', 'radiomics_robust', 'E8c_post_hoc'),
]
FUS = {}   # label -> dict(kind -> (oof, test, MCT, TACE))
for label, cnn_name, rad_name, role in FUSION_SPECS:
    c_oof, c_test, c_mct, c_tace = CNN_SOURCES[cnn_name]
    r_oof, r_test, r_mct, r_tace = RAD[rad_name]
    oof = oof_meta.copy()
    oof[W4] = c_oof.loc[oof['patient_id']].to_numpy()
    oof[W5] = r_oof.loc[oof['patient_id']].to_numpy()
    frames = {'test': pd.DataFrame(index=test_ids), 'MCT': pd.DataFrame(index=MCT_IDS), 'TACE': pd.DataFrame(index=TACE_IDS)}
    for key, (cp, rp) in {'test': (c_test, r_test), 'MCT': (c_mct, r_mct), 'TACE': (c_tace, r_tace)}.items():
        frames[key][W4] = cp.loc[frames[key].index].to_numpy()
        frames[key][W5] = rp.loc[frames[key].index].to_numpy()
    frames['test']['age'], frames['test']['gender'] = test_meta['age'].to_numpy(), test_meta['gender'].to_numpy()
    frames['MCT']['age'], frames['MCT']['gender'] = AGE_MCT, SEX_MCT
    frames['TACE']['age'], frames['TACE']['gender'] = TACE_CLIN['age'].to_numpy(), TACE_CLIN['sex'].to_numpy()
    FUS[label] = {}
    for kind, columns in {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}.items():
        fitted, oof_p = meta_folds(columns, oof)
        FUS[label][kind] = (oof_p.loc[dev_ids], apply_meta(fitted, frames['test'], columns), apply_meta(fitted, frames['MCT'], columns),
                            apply_meta(fitted, frames['TACE'], columns))
ref_internal = {k: round(macro_auc(y_test, FUS['[ref] phase_dropout_CNN + radiomics_noshape'][k][1].to_numpy()), 3) for k in ['cnn_plus_radiomics', 'full_fusion']}
GATES['reference_fusions_internal'] = ref_internal
print('Reference fusions, held-out macro-AUC (must equal E3: 0.952 and 0.951):', ref_internal)
if IN_COLAB and (abs(ref_internal['cnn_plus_radiomics'] - 0.952) > 0.002 or abs(ref_internal['full_fusion'] - 0.951) > 0.002):
    raise RuntimeError('STOP — reference fusions do not reproduce E3. Send this output.')

In [ ]:
# Cell 11 — E9a: header facts of the internal phases by class and the technical-only check (no model of images)
table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
table = table[table['patient_id'].isin(dev_ids + test_ids)].drop_duplicates(['patient_id', 'phase'])
def header_shape(row):
    shape = nib.load(str(DATA_DIR / row['ct_path'])).shape
    return row['patient_id'], row['phase'], tuple(int(s) for s in shape[:3])
with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    shapes = list(pool.map(header_shape, [r for _, r in table.iterrows()]))
H = pd.DataFrame([{'patient_id': p, 'phase': ph, 'n_i': s[0], 'n_j': s[1], 'n_slices': s[2]} for p, ph, s in shapes])
H['class'] = H['patient_id'].map(lambda p: CLASSES[LABEL[p]])
H.to_csv(OUT_DIR / 'E9a_internal_header_shapes.csv', index=False)
slices = H.pivot(index='patient_id', columns='phase', values='n_slices')
E9A = {'median_slices_by_class_phase': H.groupby(['class', 'phase'])['n_slices'].median().unstack().to_dict(),
       'median_P_over_C2_slices_by_class': (slices['P'] / slices['C2']).groupby(lambda p: CLASSES[LABEL[p]]).median().round(3).to_dict(),
       'in_plane_sizes': {f'{size}_{phase}': v for (size, phase), v in                   # flat keys (JSON cannot store pairs)
                          H.groupby(['class', 'phase'])[['n_i', 'n_j']].median().unstack().to_dict().items()}}
trunc = pd.read_csv(INPUTS2['e47_truncation'][0], dtype={'patient_id': str})
full = trunc.pivot(index='patient_id', columns='phase', values='full_voxels')
liver_dice = {p: float(2 * (LIVER_CROP[p][0] & LIVER_CROP[p][2]).sum() / max(1, LIVER_CROP[p][0].sum() + LIVER_CROP[p][2].sum())) for p in dev_ids + test_ids}
ids_all = dev_ids + test_ids
TECH = pd.DataFrame({'liver_overlap_P_vs_C2': [liver_dice[p] for p in ids_all], 'tumour_ratio_P_over_C2': [full.loc[p, 'P'] / full.loc[p, 'C2'] for p in ids_all],
                     'slice_ratio_P_over_C2': [slices.loc[p, 'P'] / slices.loc[p, 'C2'] for p in ids_all]}, index=ids_all)
y_chcc = np.array([LABEL[p] == 2 for p in ids_all], dtype=int)
cv_scores = np.zeros(len(ids_all))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(TECH, y_chcc):
    m = LogisticRegression(max_iter=1000).fit(StandardScaler().fit(TECH.iloc[tr]).transform(TECH.iloc[tr]), y_chcc[tr])
    cv_scores[va] = m.predict_proba(StandardScaler().fit(TECH.iloc[tr]).transform(TECH.iloc[va]))[:, 1]
E9A['technical_only_cv_auc_chcc_vs_rest'] = float(roc_auc_score(y_chcc, cv_scores))
E9A['single_measure_auc'] = {c: float(max(roc_auc_score(y_chcc, TECH[c]), 1 - roc_auc_score(y_chcc, TECH[c]))) for c in TECH.columns}
TECH.assign(chcc=y_chcc).to_csv(OUT_DIR / 'E9a_technical_measures.csv')
print('E9a header facts:', json.dumps({k: v for k, v in E9A.items() if k != 'in_plane_sizes'}, indent=1, default=float))

In [ ]:
# Cell 12 — Results (internal, MCT-LTDiag 203, HCC-TACE-Seg), the E9 rule, saved outputs and the block to send back
def exact_ci(k, n):
    from scipy.stats import beta
    low = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return low, high

def compute_midrank(values):
    order = np.argsort(values)
    sorted_values = values[order]
    n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64)
    index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1
        index = end
    output = np.empty(n_values, dtype=np.float64)
    output[order] = midranks
    return output

def delong_components(y, s):
    pos, neg = s[y == 1], s[y == 0]
    m, n = len(pos), len(neg)
    tx, ty, tz = compute_midrank(pos), compute_midrank(neg), compute_midrank(np.concatenate([pos, neg]))
    return tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n, (tz[:m] - tx) / n, 1.0 - (tz[m:] - ty) / m

def delong_ci(y, s):
    auc, v01, v10 = delong_components(y, s)
    se = np.sqrt(np.var(v01, ddof=1) / len(v01) + np.var(v10, ddof=1) / len(v10))
    return float(auc), float(max(0, auc - 1.96 * se)), float(min(1, auc + 1.96 * se))

def delong_paired_p(y, s1, s2):
    a1, v01a, v10a = delong_components(y, s1)
    a2, v01b, v10b = delong_components(y, s2)
    cov = np.cov(np.vstack([v01a, v01b])) / len(v01a) + np.cov(np.vstack([v10a, v10b])) / len(v10a)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    return 1.0 if var <= 0 else float(2 * norm.sf(abs(a1 - a2) / np.sqrt(var)))

rng = np.random.default_rng(SEED)
BOOT_T = [rng.integers(0, len(test_ids), len(test_ids)) for _ in range(2000)]
BOOT_M = [rng.integers(0, len(MCT_IDS), len(MCT_IDS)) for _ in range(2000)]
def hcc_score(p):
    p = p.to_numpy()
    return p[:, 0] / np.clip(p[:, 0] + p[:, 1], 1e-12, None), p.argmax(axis=1)

# ---- internal ----
INTERNAL_ITEMS = {f'CNN {name} [{pattern}]': v for (name, pattern), v in INT.items()}
INTERNAL_ITEMS.update({f'radiomics {name}': (v[0], v[1]) for name, v in RAD.items() if name in ('radiomics_noshape', 'radiomics_noP', 'radiomics_robust')})
INTERNAL_ITEMS.update({f'{kind}: {label}': (v[kind][0], v[kind][1]) for label, v in FUS.items() for kind in v})
rows = []
for name, (oof, test) in INTERNAL_ITEMS.items():
    p = test.loc[test_ids].to_numpy()
    pred = p.argmax(axis=1)
    vals = [macro_auc(y_test[b], p[b]) for b in BOOT_T if len(np.unique(y_test[b])) == 3]
    fold = [macro_auc([LABEL[x] for x in fs['val_ids']], oof.loc[fs['val_ids']].to_numpy()) for fs in fold_splits]
    rows.append({'model': name, 'test_macro_auc': macro_auc(y_test, p), 'boot_low': float(np.percentile(vals, 2.5)), 'boot_high': float(np.percentile(vals, 97.5)),
                 'cv_mean': float(np.mean(fold)), 'cv_sd': float(np.std(fold)),
                 **{f'recall_{c}': float(((pred == i) & (y_test == i)).sum() / (y_test == i).sum()) for i, c in enumerate(CLASSES)}})
INTERNAL_RESULTS = pd.DataFrame(rows)
INTERNAL_RESULTS.to_csv(OUT_DIR / 'E8b_E9_internal_results.csv', index=False)

# ---- E9 rule (fixed in the plan) ----
ir = INTERNAL_RESULTS.set_index('model')
E9_RULE = {}
for with_p, without_p in [('CNN phase_dropout_CNN [complete]', 'CNN noP_CNN [noP]'), ('radiomics radiomics_noshape', 'radiomics radiomics_noP'),
                          ('cnn_plus_radiomics: [ref] phase_dropout_CNN + radiomics_noshape', 'cnn_plus_radiomics: E9: noP_CNN + radiomics_noP'),
                          ('full_fusion: [ref] phase_dropout_CNN + radiomics_noshape', 'full_fusion: E9: noP_CNN + radiomics_noP')]:
    d_auc = float(ir.loc[without_p, 'test_macro_auc'] - ir.loc[with_p, 'test_macro_auc'])
    d_chcc = float(ir.loc[without_p, 'recall_CHCC'] - ir.loc[with_p, 'recall_CHCC'])
    E9_RULE[without_p] = {'delta_test_macro_auc': d_auc, 'delta_recall_CHCC': d_chcc,
                          'depends_on_artefact': bool(d_auc < -0.03 or d_chcc <= -0.15)}

# ---- MCT-LTDiag 203 and HCC-TACE-Seg ----
EXTERNAL_ITEMS = {f'CNN {k}': (EXT_MCT[k], EXT_TACE[k]) for k in EXT_MCT}
EXTERNAL_ITEMS.update({f'radiomics {k}': (v[2], v[3]) for k, v in RAD.items()})
EXTERNAL_ITEMS.update({f'{kind}: {label}': (v[kind][2], v[kind][3]) for label, v in FUS.items() for kind in v})
rows, tace_rows = [], []
for name, (mct, tace) in EXTERNAL_ITEMS.items():
    s, pred = hcc_score(mct.loc[MCT_IDS])
    auc, lo, hi = delong_ci(Y_MCT, s)
    vals = [roc_auc_score(Y_MCT[b], s[b]) for b in BOOT_M if 0 < Y_MCT[b].sum() < len(b)]
    rows.append({'model': name, 'auc': auc, 'delong_low': lo, 'delong_high': hi, 'boot_low': float(np.percentile(vals, 2.5)),
                 'boot_high': float(np.percentile(vals, 97.5)), 'hcc_sensitivity': float(((pred == 0) & (Y_MCT == 1)).sum() / Y_MCT.sum()),
                 'icc_specificity': float(((pred == 1) & (Y_MCT == 0)).sum() / (1 - Y_MCT).sum())})
    for subset, ids in [('clean_89', TACE_CLEAN), ('all_103', TACE_IDS)]:
        tp = tace.loc[ids].to_numpy().argmax(axis=1)
        k = int((tp == 0).sum())
        tace_rows.append({'model': name, 'subset': subset, 'hcc_correct': k, 'n': len(ids), 'hcc_sensitivity': k / len(ids),
                          'ci_low': exact_ci(k, len(ids))[0], 'ci_high': exact_ci(k, len(ids))[1],
                          'pred_ICC': int((tp == 1).sum()), 'pred_CHCC': int((tp == 2).sum())})
MCT_RESULTS, TACE_RESULTS = pd.DataFrame(rows), pd.DataFrame(tace_rows)
MCT_RESULTS.to_csv(OUT_DIR / 'E8b_E9_mct203_results.csv', index=False)
TACE_RESULTS.to_csv(OUT_DIR / 'E8b_E9_hcc_tace_seg_results.csv', index=False)

PAIRS = [('CNN phase_dropout_CNN + harmonisation', 'CNN phase_dropout_CNN'), ('CNN DR_CNN', 'CNN phase_dropout_CNN'),
         ('CNN DR_CNN + harmonisation', 'CNN phase_dropout_CNN'), ('CNN noP_CNN', 'CNN phase_dropout_CNN'),
         ('radiomics radiomics_noP', 'radiomics radiomics_noshape')]
for kind in ['cnn_plus_radiomics', 'full_fusion']:
    ref = f'{kind}: [ref] phase_dropout_CNN + radiomics_noshape'
    PAIRS += [(f'{kind}: {label}', ref) for label, _, _, role in FUSION_SPECS if role != 'reference']
pair_rows = []
for a, b in PAIRS:
    sa, _ = hcc_score(EXTERNAL_ITEMS[a][0].loc[MCT_IDS]); sb, _ = hcc_score(EXTERNAL_ITEMS[b][0].loc[MCT_IDS])
    diffs = [roc_auc_score(Y_MCT[i], sa[i]) - roc_auc_score(Y_MCT[i], sb[i]) for i in BOOT_M if 0 < Y_MCT[i].sum() < len(i)]
    pair_rows.append({'model': a, 'reference': b, 'auc': roc_auc_score(Y_MCT, sa), 'auc_reference': roc_auc_score(Y_MCT, sb),
                      'difference': roc_auc_score(Y_MCT, sa) - roc_auc_score(Y_MCT, sb), 'boot_low': float(np.percentile(diffs, 2.5)),
                      'boot_high': float(np.percentile(diffs, 97.5)), 'delong_p': delong_paired_p(Y_MCT, sa, sb)})
PAIRED = pd.DataFrame(pair_rows)
PAIRED.to_csv(OUT_DIR / 'E8b_E9_mct203_paired_differences.csv', index=False)

long = []
for name, (mct, tace) in EXTERNAL_ITEMS.items():
    for cohort, frame in [('MCT_LTDiag', mct), ('HCC_TACE_Seg', tace)]:
        t = frame.copy(); t.insert(0, 'patient_id', t.index); t.insert(0, 'cohort', cohort); t.insert(0, 'model', name); long.append(t)
for name, (oof, test) in INTERNAL_ITEMS.items():
    for split_name, frame in [('oof', oof), ('test', test)]:
        t = frame.copy(); t.insert(0, 'patient_id', t.index); t.insert(0, 'cohort', f'internal_{split_name}'); t.insert(0, 'model', name); long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E8b_E9_predictions.csv', index=False)

summary = {'run_id': RUN_ID, 'notebook': 'E8b_E9_CNN_mitigation_and_noncontrast_check.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes,
           'gates': GATES,
           'E9a': E9A, 'E9_rule': E9_RULE, 'epochs': NUM_EPOCHS,
           'protocol': '00 log sections 21 (E8b, E8c) and 22 (E9); internal training only; each external evaluation once; primary results unchanged'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E8b_{RUN_ID} | epochs {NUM_EPOCHS} | gates: {GATES}')
print('E9a header facts (median slices P/C2 by class):', E9A['median_P_over_C2_slices_by_class'],
      '| technical-only CV AUC cHCC-CCA vs rest:', round(E9A['technical_only_cv_auc_chcc_vs_rest'], 3), '| single measures:', {k: round(v, 3) for k, v in E9A['single_measure_auc'].items()})
print('\n--- internal held-out 56 (macro-AUC, bootstrap CI; CV mean ± SD; recalls) ---')
print(INTERNAL_RESULTS.round(3).to_string(index=False))
print('\n--- E9 rule (without the non-contrast phase vs with it) ---')
print(pd.DataFrame(E9_RULE).T.round(3).to_string())
print('\n--- MCT-LTDiag 203, HCC vs ICC ---')
print(MCT_RESULTS.round(3).to_string(index=False))
print('\n--- paired AUC differences vs the reference (bootstrap 95% CI, DeLong p) ---')
print(PAIRED.round(3).to_string(index=False))
print('\n--- HCC-TACE-Seg corrected, clean 89 ---')
print(TACE_RESULTS[TACE_RESULTS['subset'] == 'clean_89'][['model', 'hcc_correct', 'n', 'hcc_sensitivity', 'ci_low', 'ci_high']].round(3).to_string(index=False))
print('\nVersions:', VERSIONS)
print('==============================================================')

In [ ]:
# Cell 13 — Finish: make sure every result file has reached Google Drive, then shut the runtime down (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime shuts down in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()